# 🏥 Kaggle ViHaluEval 40,000 QA Medical Dataset Generator
## 🚀 Clean High-Throughput Medical QA Generator
**Source Data**: `duoc_thu_2018_cleaned.txt` (14.1 Million Chars / 1,026 Drug Monographs)
**Output**: `vietnamese_medical_halueval_40k_full.json` (40,000 High-Quality QA Pairs)

In [1]:
# Cell 1: Install vLLM Engine
!pip install -q vllm tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.7/303.7 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.8/

In [2]:
# Cell 2: Setup Environment
import os
import re
import json
import glob
import torch
from tqdm.auto import tqdm

print(f"✅ Environment Ready. Detected {torch.cuda.device_count()} GPU(s).")

✅ Environment Ready. Detected 2 GPU(s).


In [3]:
# Cell 3: Load Qwen2.5-7B-Instruct AWQ Model Engine
from vllm import LLM, SamplingParams

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-AWQ"

if 'llm' not in globals():
    print(f"🚀 Loading vLLM engine for {MODEL_NAME}...")
    llm = LLM(
        model=MODEL_NAME,
        quantization="awq",
        tensor_parallel_size=1,
        gpu_memory_utilization=0.75,
        max_model_len=3072,
        trust_remote_code=True
    )
    print("✅ vLLM Engine Loaded Successfully!")
else:
    print("⚡ vLLM Engine already in memory.")

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=650
)

🚀 Loading vLLM engine for Qwen/Qwen2.5-7B-Instruct-AWQ...
INFO 08-09 17:36:26 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'max_model_len': 3072, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'quantization': 'awq', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 08-09 17:36:41 [model.py:623] Resolved architecture: Qwen2ForCausalLM
INFO 08-09 17:36:41 [model.py:1788] Using max model len 3072
INFO 08-09 17:36:42 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

...

(EngineCore pid=200) INFO 08-09 17:36:50 [model_runner.py:284] Loading model from scratch...
(EngineCore pid=200) INFO 08-09 17:36:50 [auto_awq.py:473] Using MarlinLinearKernel for AutoAWQMarlinLinearMethod
(EngineCore pid=200) ERROR 08-09 17:36:50 [fa_utils.py:253] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(EngineCore pid=200) INFO 08-09 17:36:52 [cuda.py:482] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=200) INFO 08-09 17:37:17 [weight_utils.py:530] Time spent downloading weights for Qwen/Qwen2.5-7B-Instruct-AWQ: 24.242904 seconds
(EngineCore pid=200) INFO 08-09 17:37:17 [weight_utils.py:869] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 5.19 GiB. Available RAM: 28.30 GiB.
(EngineCore pid=200) INFO 08-09 17:37:17 [weight_utils.py:892] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre).

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=200) INFO 08-09 17:37:19 [default_loader.py:430] Loading weights took 1.93 seconds
(EngineCore pid=200) INFO 08-09 17:37:22 [model_runner.py:305] Model loading took 5.29 GiB and 32.417168 seconds
(EngineCore pid=200) WARNING 08-09 17:37:22 [topk_topp_sampler.py:62] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=200) INFO 08-09 17:37:35 [backends.py:1094] Using cache directory: /root/.cache/vllm/torch_compile_cache/a1733f46a6/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=200) INFO 08-09 17:37:35 [backends.py:1155] Dynamo bytecode transform time: 13.04 s
(EngineCore pid=200) INFO 08-09 17:37:41 [backends.py:378] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=200) INFO 08-09 17:37:48 [backends.py:393] Compiling a graph for compile range (1, 8192) takes 11.28 s
(EngineCore pid=200) INFO 08-09 17:37:54 [decorators.py:708] saved 

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:09<00:00,  3.62it/s]


(EngineCore pid=200) INFO 08-09 17:38:20 [model_runner.py:747] Graph capturing finished in 19 secs, took 0.52 GiB
(EngineCore pid=200) INFO 08-09 17:38:20 [gpu_worker.py:857] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.75, 10.92 GiB). Actual usage is 5.29 GiB for weight, 1.13 GiB for peak activation, 0.05 GiB for non-torch memory, and 0.52 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=4076135424` (3.8 GiB) to fit into requested memory, or `--kv-cache-memory=7875175424` (7.33 GiB) to fully utilize gpu memory. Current kv cache memory in use is 4.46 GiB.
(EngineCore pid=200) INFO 08-09 17:38:37 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=200) INFO 08-09 17:38:37 [core.py:340] init engine (profile, create kv cache, warmup model) took 75.45 s (compilation: 31.35 s)
(EngineCore pid=200) INFO 08-09 17:38:37 [kernel.py:295]

In [4]:
# Cell 4: Automatic Dataset Path Finding & Chunking (~40,000 QA target)
search_paths = [
    "duoc_thu_2018_cleaned.txt",
    "/kaggle/input/datasets/tunthanh66/duoc-thu-2018/duoc_thu_2018_cleaned.txt"
]

DATA_PATH = None
for p in search_paths:
    matches = glob.glob(p, recursive=True)
    if matches:
        DATA_PATH = matches[0]
        break

if DATA_PATH is None:
    raise FileNotFoundError("❌ Không tìm thấy file 'duoc_thu_2018_cleaned.txt'. Vui lòng Upload file vào Kaggle Input!")

print(f"📂 Found Data File at: {DATA_PATH}")
with open(DATA_PATH, "r", encoding="utf-8", errors="ignore") as f:
    raw_lines = f.readlines()

chunks = []
current_chunk = []
current_len = 0

for line in raw_lines:
    l = line.strip()
    if not l:
        continue
    current_chunk.append(l)
    current_len += len(l)
    if current_len >= 1000:
        chunks.append("\n".join(current_chunk))
        current_chunk = []
        current_len = 0

if current_chunk:
    chunks.append("\n".join(current_chunk))

print(f"📊 Total Context Chunks Created: {len(chunks)}")
print(f"🎯 Target QA Generation: {len(chunks) * 3} samples (~40,000 QA Pairs)")

📂 Found Data File at: /kaggle/input/datasets/tunthanh66/duoc-thu-2018/duoc_thu_2018_cleaned.txt
📊 Total Context Chunks Created: 11455
🎯 Target QA Generation: 34365 samples (~40,000 QA Pairs)


In [5]:
# Cell 5: Prompt Template for 3 QA Pairs
PROMPT_TEMPLATE = """<|im_start|>system
Bạn là chuyên gia Dược học & Đánh giá Ảo giác LLM Y tế hàng đầu Việt Nam. Dựa vào ngữ cảnh Dược thư được cung cấp, hãy tạo đúng 3 cặp QA (Câu hỏi + Câu trả lời) gồm:
1. [CHUẨN XÁC]: Trả lời chính xác 100% theo ngữ cảnh.
2. [ẢO GIÁC - SAI LIỀU/TƯƠNG TÁC]: Trả lời chứa ảo giác sai lệch về liều dùng hoặc tương tác thuốc.
3. [ẢO GIÁC - MÂU THUẪN CHỐNG CHỈ ĐỊNH]: Trả lời chứa thông tin chống chỉ định sai hoàn toàn.

Trả về định dạng JSON duy nhất:
[
  {{
    "question": "Câu hỏi y tế 1...",
    "right_answer": "Câu trả lời đúng...",
    "hallucinated_answer": "Câu trả lời ảo giác...",
    "hallucination_type": "misleading_dosage"
  }},
  {{
    "question": "Câu hỏi y tế 2...",
    "right_answer": "...",
    "hallucinated_answer": "...",
    "hallucination_type": "contradictory_contraindication"
  }},
  {{
    "question": "Câu hỏi y tế 3...",
    "right_answer": "...",
    "hallucinated_answer": "...",
    "hallucination_type": "factual_error"
  }}
]<|im_end|>
<|im_start|>user
NGỮ CẢNH DƯỢC THƯ:
{context}<|im_end|>
<|im_start|>assistant
"""

In [6]:
# Cell 6: Universal Standard Batch Generation & Auto-Checkpointing
BATCH_SIZE = 64
OUTPUT_FILE = "vietnamese_medical_halueval_40k_full.json"
CHECKPOINT_INTERVAL = 5000

all_dataset = []
print("🚀 STARTING 40,000 QA BATCH GENERATION...")

for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Batch Generation"):
    batch_chunks = chunks[i:i+BATCH_SIZE]
    prompts = [PROMPT_TEMPLATE.format(context=c[:1200]) for c in batch_chunks]
    
    # Standard universal vLLM call signature (works across all vLLM versions)
    outputs = llm.generate(prompts, sampling_params)
    
    for ctx, out in zip(batch_chunks, outputs):
        text_resp = out.outputs[0].text.strip()
        try:
            json_match = re.search(r'\[.*?\]', text_resp, re.DOTALL)
            if json_match:
                qa_items = json.loads(json_match.group(0))
                for item in qa_items:
                    all_dataset.append({
                        "id": f"med_40k_{len(all_dataset)+1:05d}",
                        "knowledge_context": ctx[:800],
                        "question": item.get("question", ""),
                        "right_answer": item.get("right_answer", ""),
                        "hallucinated_answer": item.get("hallucinated_answer", ""),
                        "hallucination_type": item.get("hallucination_type", "misleading_dosage"),
                        "domain": "Medical_Pharmacopoeia"
                    })
        except Exception as e:
            pass
            
    if len(all_dataset) >= CHECKPOINT_INTERVAL and (len(all_dataset) % CHECKPOINT_INTERVAL) < (BATCH_SIZE * 3):
        ckpt_path = f"vietnamese_medical_halueval_40k_ckpt_{len(all_dataset)}.json"
        with open(ckpt_path, "w", encoding="utf-8") as f:
            json.dump(all_dataset, f, ensure_ascii=False, indent=2)
        print(f"💾 Saved Checkpoint: {len(all_dataset)} samples to {ckpt_path}")

# Final Save
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_dataset, f, ensure_ascii=False, indent=2)

print(f"🎉 SUCCESS: Generated {len(all_dataset)} Medical QA samples saved to {OUTPUT_FILE}!")

🚀 STARTING 40,000 QA BATCH GENERATION...


Batch Generation:   0%|          | 0/179 [00:00<?, ?it/s]

Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 64/64 [01:01<00:00,  1.05it/s, est. speed input: 723.74 toks/s, output: 365.12 toks/s]


Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 64/64 [01:12<00:00,  1.13s/it, est. speed input: 615.90 toks/s, output: 319.56 toks/s]


Rendering prompts:   0%|          | 0/63 [00:00<?, ?it/s]


Processed prompts: 100%|██████████| 63/63 [01:04<00:00,  1.02s/it, est. speed input: 723.73 toks/s, output: 303.97 toks/s]


🎉 SUCCESS: Generated 33927 Medical QA samples saved to vietnamese_medical_halueval_40k_full.json!
